# 10 · Practical E — Constructing an ATM Curve

**Read first:** Chapter 11 (*ATM Curve Construction*), then Practical E.

This is the longest stage and the one that most changes how you read a volatility screen. Budget six hours.

---

## What you'll be able to do after this

- Explain why volatility curves are built in **variance** and quoted in volatility, and what breaks when you forget.
- Produce the ATM saw-tooth from first principles, and say exactly which weekend produced which tooth.
- Weight an event date and read the forward overnight volatility strip a trader actually looks at.

## The intuition, before the maths

### One idea: uncertainty accumulates, and it never un-accumulates

Volatility is not a thing that adds up. **Variance is.**

If Monday might move spot by some amount, and Tuesday might too, the uncertainty over both days is the *sum* of the two daily uncertainties — measured in variance. Not in volatility. That single fact drives the entire chapter.

Two properties follow, and they are the whole toolkit:

1. **Variance can never be negative.** You cannot become *less* uncertain about the future by waiting longer.
2. **Variance adds across time.** Variance over two days = variance on day one + variance on day two.

Volatility has neither property. It is variance divided by time and square-rooted — a *rate*, not a quantity. Adding volatilities is like adding speeds and expecting a distance.

### The observation the whole practical explains

Look at a real short-dated volatility screen and you see a saw-tooth. Friday expiries price higher than the Monday expiries around them, over and over.

Why? A one-week option contains **five trading days and two closed ones**. The market cannot move at the weekend. But the Black-Scholes formula does not know that — it is fed `T` in calendar days, and calendar days include Saturday.

So the market has an accounting problem: variance accumulates only on open days, but time is quoted in calendar days. Divide one by the other and Monday expiries — which have just absorbed two dead days — come out lower.

That mismatch *is* the saw-tooth. It is not a market inefficiency, it is the market correctly pricing the fact that nothing happens on Sunday.

### A worked example with round numbers

Overnight volatility is 12%. Assume every weekday is equally uncertain and the weekend is dead.

A one-week option spans 7 calendar days but only **5** live ones:

```
variance over the week = 5 × (one day's variance)
quoted against         = 7 calendar days
so                     σ_1wk = 12% × √(5/7) = 10.1%
```

The one-week volatility is **lower than the overnight**, despite covering more time. Nothing about the market changed — only the ratio of open days to calendar days.

That is the whole of Task C in one calculation.

## The maths, derived not asserted

### Variance

$$\text{var}(T) = \sigma^2 T$$

Chapter 11's example: a 3-month option at 12% has variance $0.12^2 \times 0.25 = 0.0036$.

### Forward volatility

Variance being additive means the variance *between* two future dates is just a subtraction:

$$\sigma_{\text{fwd}} = \sqrt{\frac{\sigma_2^2 T_2 - \sigma_1^2 T_1}{T_2 - T_1}}$$

If that numerator ever goes negative you have an **arbitrage**, not a rounding error. It says the market thinks a later date is *less* uncertain than an earlier one, which cannot be true — sell the near option, buy the far one, and you are short variance for free.

### Two ways to interpolate, neither of them right

$$\text{linear vol:}\quad \sigma(t) = \sigma_1 + (\sigma_2 - \sigma_1)\tfrac{t - T_1}{T_2 - T_1}$$
$$\text{linear variance:}\quad \text{var}(t) = \text{var}_1 + (\text{var}_2 - \text{var}_1)\tfrac{t - T_1}{T_2 - T_1},\quad \sigma(t) = \sqrt{\text{var}(t)/t}$$

- **Linear volatility** looks right and *can produce negative forward variance from perfectly valid inputs.* Experiment 1 does it.
- **Linear variance** is safe and looks wrong — daily variance jumps discontinuously at every tenor date, and there is no reason the day before the 3-month tenor should differ from the day after.

Chapter 11 is candid that desks use a combination and control daily variance more carefully than either pure method does.

### The parametric model

$$\sigma_T = \sigma_{\text{short}} + (\sigma_{\text{long}} - \sigma_{\text{short}})(1 - e^{-\lambda T})$$

The book says outright this form **would never be used in practice** because nothing in it enforces non-negative forward variance. It is here to make the shape of a term structure concrete.

### Day weights — the mechanism of Task C

Every calendar date gets a weight. Then:

$$\text{economic time}(t) = \frac{\sum_{i \le t} \omega_i}{365}, \qquad \text{var}(t) = \sigma^2 \sum_{i \le t} \omega_i \, dt, \quad dt = \tfrac{1}{365}$$

and — **this is the part that matters** —

$$\sigma_{\text{ATM}}(t) = \sqrt{\frac{\text{var}(t)}{\text{calendar time}(t)}}$$

**Variance accumulates in economic time. Volatility is quoted against calendar time.** The asymmetry between those two lines is the entire trick. Conflate them and the saw-tooth vanishes, because you have quietly assumed the market is open at the weekend.

## The code

In [1]:
from datetime import date, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fxds.atm_curve import (
    ATMCurve, Interpolation, CurveRangeError,
    ParametricATMCurve, calibrate_parametric,
    WeightedATMCurve, DEFAULT_DAY_WEIGHTS, WEEKEND_ZERO_WEIGHTS,
    variance, volatility_from_variance, forward_volatility,
)
from fxds.dates import expiry_from_tenor
from fxds.plotting import (
    use_house_style, style_axis, mark_level, as_percent,
    PRIMARY, SECONDARY, TERTIARY, QUATERNARY, MUTED, ALERT,
)

use_house_style()
pd.set_option("display.precision", 6)

HORIZON = date(2014, 6, 11)
TENORS = ("1W", "1M", "3M", "6M", "1Y", "2Y")

### Variance first — the two properties everything rests on

In [2]:
print(f"3mth ATM at 12%:  variance = 0.12^2 x 0.25 = {variance(0.12, 0.25):.4f}")
print(f"  (Ch. 11's worked example)\n")

# Additivity, demonstrated.
v1, T1, v2, T2 = 0.105, 0.5, 0.117, 1.0
fwd = forward_volatility(v1, T1, v2, T2)
print(f"6mth ATM {v1:.1%}, 1yr ATM {v2:.1%}")
print(f"  forward volatility 6mth->1yr = {fwd:.2%}   (Ch. 11: 12.8%)")
print(f"  check: var(6m) + var(fwd over 6m) = {variance(v1,T1) + variance(fwd, T2-T1):.6f}")
print(f"         var(1y)                    = {variance(v2,T2):.6f}   <- identical")

3mth ATM at 12%:  variance = 0.12^2 x 0.25 = 0.0036
  (Ch. 11's worked example)

6mth ATM 10.5%, 1yr ATM 11.7%
  forward volatility 6mth->1yr = 12.79%   (Ch. 11: 12.8%)
  check: var(6m) + var(fwd over 6m) = 0.013689
         var(1y)                    = 0.013689   <- identical


Note the forward volatility (12.8%) is **higher than either quoted level**. The 1-year contains the 6-month, so if the whole year averages 11.7% and the first half only managed 10.5%, the second half has to make up the difference. That is what a forward volatility tells you, and it is the number a trader actually trades when they buy a calendar spread.

### Task A — interpolation, and the four query cases

In [3]:
expiries = [expiry_from_tenor(HORIZON, t) for t in TENORS]
market_vols = [0.0685, 0.0720, 0.0765, 0.0800, 0.0835, 0.0870]

curve = ATMCurve(HORIZON, expiries, market_vols, method=Interpolation.LINEAR_VARIANCE)

print("The four cases Practical E asks you to test:\n")

# 1 & 2: outside the quoted range.
for label, query in [("before first tenor", HORIZON + timedelta(days=1)),
                     ("after last tenor",   HORIZON + timedelta(days=2000))]:
    try:
        curve.volatility(query)
    except CurveRangeError as exc:
        print(f"  {label:20s} -> {type(exc).__name__}: {str(exc)[:52]}...")

# 3: exactly on a tenor.
print(f"\n  exactly on 3M        -> {curve.volatility(expiries[2]):.4%}  "
      f"(quoted: {market_vols[2]:.4%})")

# 4: between two tenors.
mid = expiries[2] + (expiries[3] - expiries[2]) / 2
print(f"  between 3M and 6M    -> {curve.volatility(mid):.4%}  "
      f"(between {market_vols[2]:.2%} and {market_vols[3]:.2%})")

The four cases Practical E asks you to test:

  before first tenor   -> CurveRangeError: Query date 2014-06-12 is before the first tenor expi...
  after last tenor     -> CurveRangeError: Query date 2019-12-02 is after the last tenor expiry...

  exactly on 3M        -> 7.6500%  (quoted: 7.6500%)
  between 3M and 6M    -> 7.8829%  (between 7.65% and 8.00%)


> The book returns `-1` outside the range. A magic number that is also a plausible volatility will propagate silently into a variance and produce something that looks like an answer — so this raises. Recorded in `notes/deviations.md`.

### The daily curve, with variance alongside

In [4]:
daily = curve.daily_curve()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))

axes[0].plot(daily["years"], daily["atm_vol"], color=PRIMARY, linewidth=2)
axes[0].scatter(curve.times, market_vols, color=ALERT, zorder=5, s=45, label="market tenors")
axes[0].legend(); as_percent(axes[0], decimals=1)
style_axis(axes[0], "ATM volatility", "Time to expiry (years)", "ATM implied volatility",
           "Red dots are the quoted tenors. Everything between them is the interpolation's opinion.")

axes[1].plot(daily["years"], daily["total_variance"], color=TERTIARY, linewidth=2)
style_axis(axes[1], "Total variance", "Time to expiry (years)", "Total variance (dimensionless)",
           "This is the quantity that must never fall. It is monotone here, so the curve is arbitrage-free.")
plt.tight_layout(); plt.show()

print(f"negative forward variance anywhere? {curve.has_negative_forward_variance()}")

negative forward variance anywhere? False


## Experiments

### Experiment 1 — Break a curve with valid inputs

Chapter 11's counterexample. ATM is a flat 20% out to one year, then 15% at two years. Every input is a perfectly reasonable volatility.

**Predict:** interpolate linearly in volatility. Is the result arbitrage-free?

In [5]:
bad_vols = [0.20, 0.20, 0.20, 0.20, 0.20, 0.15]

print("The book's arithmetic:")
print(f"  variance to 1yr   at 20.0%              = {variance(0.20, 1.0):.4f}")
print(f"  variance to 18mth at 17.5% (interpolated) = {variance(0.175, 1.5):.4f}   <- HIGHER")
print(f"  variance to 2yr   at 15.0%              = {variance(0.15, 2.0):.4f}   <- then LOWER")
print("\nVariance falls between 18 months and 2 years. That is an arbitrage.\n")

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
for method, colour, name in [
    (Interpolation.LINEAR_VOLATILITY, ALERT, "linear in volatility"),
    (Interpolation.LINEAR_VARIANCE, PRIMARY, "linear in variance"),
]:
    c = ATMCurve(HORIZON, expiries, bad_vols, method=method)
    d = c.daily_curve()
    axes[0].plot(d["years"], d["atm_vol"], color=colour, linewidth=2, label=name)
    axes[1].plot(d["years"], d["total_variance"], color=colour, linewidth=2, label=name)
    print(f"  {name:22s}: negative forward variance? {c.has_negative_forward_variance()}")

axes[0].scatter(ATMCurve(HORIZON, expiries, bad_vols).times, bad_vols,
                color=MUTED, zorder=5, s=40)
axes[0].legend(); as_percent(axes[0], decimals=0)
style_axis(axes[0], "Both look plausible as volatility", "Years", "ATM implied volatility",
           "The red curve is the intuitive one. It is also the broken one.")

axes[1].legend()
style_axis(axes[1], "Only one is monotone in variance", "Years", "Total variance",
           "The red line turns over past 1.5 years. Variance falling with time is a free lunch.")
plt.tight_layout(); plt.show()

The book's arithmetic:
  variance to 1yr   at 20.0%              = 0.0400
  variance to 18mth at 17.5% (interpolated) = 0.0459   <- HIGHER
  variance to 2yr   at 15.0%              = 0.0450   <- then LOWER

Variance falls between 18 months and 2 years. That is an arbitrage.

  linear in volatility  : negative forward variance? True


  linear in variance    : negative forward variance? False


**Result:** linear-in-volatility produces the more natural-looking curve and is arbitrageable. Linear-in-variance is safe.

This is the tradeoff Chapter 11 wants you to feel. The intuitive method is unsafe; the safe method is unintuitive. Neither survives alone, which is why real desks build in variance terms and then work hard on how daily variance evolves.

### Experiment 2 — Why linear-in-variance looks wrong

**Predict:** take the *sensible* upward-sloping curve and interpolate it in variance. Look at daily variance. What shape do you expect between tenors?

In [6]:
fig, ax = plt.subplots(figsize=(11, 4.6))
for method, colour, name in [
    (Interpolation.LINEAR_VOLATILITY, SECONDARY, "linear in volatility"),
    (Interpolation.LINEAR_VARIANCE, PRIMARY, "linear in variance"),
]:
    d = ATMCurve(HORIZON, expiries, market_vols, method=method).daily_curve()
    ax.plot(d["years"], d["daily_variance"] * 365, color=colour, linewidth=1.8, label=name)

for T in ATMCurve(HORIZON, expiries, market_vols).times:
    ax.axvline(T, color=MUTED, linestyle=":", linewidth=1)

ax.legend()
style_axis(ax, "Daily variance, annualised", "Time to expiry (years)",
           "Daily variance x 365 (dimensionless)",
           "Dotted lines are tenor dates. Linear-in-variance is piecewise flat and jumps at every tenor - there is no reason the day before the 3M tenor should differ from the day after.")
plt.show()

**Result:** linear-in-variance gives a **step function** — flat between tenors, discontinuous at each one. Linear-in-volatility gives something smoothly varying but, as Experiment 1 showed, unbounded below.

Chapter 11's objection is exactly this: *"Intuitively it does not make sense that daily variance should jump immediately past each market tenor date."* Nothing happens on the 3-month tenor date to justify a discontinuity in how volatile the market expects to be.

### Experiment 3 — Fitting the parametric model

**Predict:** the three-parameter model has a fixed shape — monotone, exponentially approaching a long-term level. Fit it to the realistic upward-sloping curve, then to a curve with a kink. Which fits, and what does the residual tell you?

In [7]:
times = np.array(ATMCurve(HORIZON, expiries, market_vols).times)

smooth = calibrate_parametric(times, np.array(market_vols))
kinked_vols = np.array([0.0685, 0.0850, 0.0700, 0.0800, 0.0835, 0.0870])
kinked = calibrate_parametric(times, kinked_vols)

for name, result, target in [("realistic curve", smooth, market_vols),
                             ("curve with a kink", kinked, kinked_vols)]:
    c = result.curve
    print(f"{name}:")
    print(f"   sigma_short {c.sigma_short:.4%}   sigma_long {c.sigma_long:.4%}   "
          f"lambda {c.speed:.3f}")
    print(f"   RMSE {result.rmse:.5f}  ({result.rmse*100:.3f} vol points)   "
          f"max error {result.max_error*100:.3f} points\n")

fig, ax = plt.subplots()
grid = np.linspace(0.01, 2.2, 300)
ax.plot(grid, smooth.curve.volatility(grid), color=PRIMARY, label="fit to realistic curve")
ax.scatter(times, market_vols, color=PRIMARY, s=45, zorder=5)
ax.plot(grid, kinked.curve.volatility(grid), color=ALERT, linestyle="--",
        label="fit to kinked curve")
ax.scatter(times, kinked_vols, color=ALERT, s=45, zorder=5, marker="s")
ax.legend(); as_percent(ax, decimals=1)
style_axis(ax, "The parametric model can only bend one way",
           "Time to expiry (years)", "ATM implied volatility",
           "Squares are a market with a kink the functional form cannot reach. The residual at each tenor IS the override a trader would input.")
plt.show()

realistic curve:
   sigma_short 6.8636%   sigma_long 8.6779%   lambda 2.016
   RMSE 0.00066  (0.066 vol points)   max error 0.086 points

curve with a kink:
   sigma_short 7.3780%   sigma_long 9.2716%   lambda 0.626
   RMSE 0.00550  (0.550 vol points)   max error 1.030 points



**Result:** the smooth curve fits to well under a tenth of a vol point. The kinked one cannot be fitted at all — the model is monotone by construction and the market is not.

That failure is useful. Chapter 11 describes traders inputting **overrides** at tenors where the model misses, and notes the override is itself information: *"this suggests that the 2mth ATM is relatively cheaper than other tenors."* A least-squares residual is that override, computed rather than eyeballed.

The book fits this by hand. Adding the calibration is beyond the practical and worth the fifteen lines.

### Experiment 4 — The magic: turn the weekend off

This is the moment Practical E is built around. Do it in two stages and **predict before each**.

**Stage 1 — every weekday weighted 1.0, including the weekend.** What shape is the curve?

In [8]:
flat = WeightedATMCurve(HORIZON, flat_volatility=0.10)
flat_frame = flat.build(days=120)

print(f"weights: all 1.0")
print(f"  ATM volatility range: {flat_frame['atm_vol'].min():.4%} to "
      f"{flat_frame['atm_vol'].max():.4%}")
print(f"  standard deviation:   {flat_frame['atm_vol'].std():.2e}")
print(f"  economic time == calendar time? "
      f"{np.allclose(flat_frame['economic_time'], flat_frame['calendar_time'])}")

weights: all 1.0
  ATM volatility range: 10.0000% to 10.0000%
  standard deviation:   2.58e-17
  economic time == calendar time? True


**Stage 1 result:** perfectly flat at the input volatility, and economic time is identical to calendar time. With every day weighted equally there is no distinction to make.

**Stage 2 — set the weekend weights to zero.** Now predict: what happens to the shape, and what happens to the *level*?

In [9]:
sawtooth = WeightedATMCurve(HORIZON, 0.10, weekday_weights=dict(WEEKEND_ZERO_WEIGHTS))
saw_frame = sawtooth.build(days=120)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)

axes[0].plot(flat_frame["date"], flat_frame["atm_vol"], color=PRIMARY, linewidth=2)
as_percent(axes[0], decimals=1)
style_axis(axes[0], "Before: every day weighted 1.0", "Expiry date", "ATM implied volatility",
           "Economic time equals calendar time, so the curve is flat at the 10% input.")

axes[1].plot(saw_frame["date"], saw_frame["atm_vol"], color=SECONDARY, linewidth=1.6)
fridays = saw_frame[saw_frame["weekday"] == "Fri"]
mondays = saw_frame[saw_frame["weekday"] == "Mon"]
axes[1].scatter(fridays["date"], fridays["atm_vol"], color=TERTIARY, s=22, zorder=5, label="Friday")
axes[1].scatter(mondays["date"], mondays["atm_vol"], color=ALERT, s=22, zorder=5, label="Monday")
axes[1].legend()
style_axis(axes[1], "After: weekend weighted 0.0", "Expiry date", "",
           "The saw-tooth. Every Monday (red) prices below the Friday before it (green), because economic time stopped for two days.")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

print(f"level after 120 days: {saw_frame['atm_vol'].iloc[-1]:.4%}")
print(f"the 5/7 limit:        {0.10 * np.sqrt(5/7):.4%}")

level after 120 days: 8.4656%
the 5/7 limit:        8.4515%


**Stage 2 result:** the saw-tooth appears, and the **level drops** — tending toward `10% × √(5/7) = 8.45%` rather than staying at 10%.

Both effects come from the same place. The economic-to-calendar time ratio is now 5/7, so:

- **The shape**: each Monday has just absorbed two zero-variance days, so its ratio is at a local minimum. Friday's is at a local maximum. Hence the tooth.
- **The level**: over any long window the ratio settles at 5/7, and volatility is the square root of variance over calendar time — so the whole curve sits at √(5/7) of the input.

Chapter 11 notes desks adjust for the level effect when they have target volatilities to hit, and that in practice the weekend gets a **small non-zero** weight rather than exactly zero — because weekend news can gap spot on the Monday open. Exactly zero is the practical's simplification and it is the version that shows the effect most clearly.

Let me show the mechanism directly in the numbers.

In [10]:
window = saw_frame[(saw_frame["date"] >= date(2014, 6, 19)) &
                   (saw_frame["date"] <= date(2014, 6, 26))]
print(window[["date", "weekday", "weight", "calendar_time", "economic_time",
              "total_variance", "atm_vol"]].to_string(index=False))
print("\nWatch total_variance: FLAT across Sat and Sun - no variance accumulates.")
print("But calendar_time keeps rising. Volatility = sqrt(variance / calendar time),")
print("so dividing a flat numerator by a growing denominator drags the ATM down.")
print("Monday adds variance again and the tooth turns back up.")

      date weekday  weight  calendar_time  economic_time  total_variance  atm_vol
2014-06-19     Thu     1.0       0.021918       0.016438        0.000164 0.086603
2014-06-20     Fri     1.0       0.024658       0.019178        0.000192 0.088192
2014-06-21     Sat     0.0       0.027397       0.019178        0.000192 0.083666
2014-06-22     Sun     0.0       0.030137       0.019178        0.000192 0.079772
2014-06-23     Mon     1.0       0.032877       0.021918        0.000219 0.081650
2014-06-24     Tue     1.0       0.035616       0.024658        0.000247 0.083205
2014-06-25     Wed     1.0       0.038356       0.027397        0.000274 0.084515
2014-06-26     Thu     1.0       0.041096       0.030137        0.000301 0.085635

Watch total_variance: FLAT across Sat and Sun - no variance accumulates.
But calendar_time keeps rising. Volatility = sqrt(variance / calendar time),
so dividing a flat numerator by a growing denominator drags the ATM down.
Monday adds variance again and the to

### Experiment 5 — Weight an event date

Chapter 11: event days get higher expected variance, which raises the ATM for that expiry **and every expiry after it**. The book uses Non-Farm Payrolls on Thursday 3 July 2014.

**Predict:** raise the weight on that one date. Which expiries move — only that one, only later ones, or all of them?

In [11]:
NFP = date(2014, 7, 3)

event = WeightedATMCurve(HORIZON, 0.10, weekday_weights=dict(WEEKEND_ZERO_WEIGHTS))
event.set_event(NFP, 4.0)     # four normal days' worth of variance on one date
event_frame = event.build(days=120)

merged = saw_frame[["date", "weekday", "atm_vol"]].merge(
    event_frame[["date", "atm_vol"]], on="date", suffixes=("_base", "_event"))
merged["change"] = merged["atm_vol_event"] - merged["atm_vol_base"]

before = merged[merged["date"] < NFP]
after = merged[merged["date"] >= NFP]
print(f"expiries BEFORE the event that moved: "
      f"{(before['change'].abs() > 1e-12).sum()} of {len(before)}")
print(f"expiries FROM the event onward that ROSE: "
      f"{(after['change'] > 1e-12).sum()} of {len(after)}")

fig, axes = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(merged["date"], merged["atm_vol_base"], color=MUTED, linewidth=1.5,
             label="no event")
axes[0].plot(merged["date"], merged["atm_vol_event"], color=PRIMARY, linewidth=1.8,
             label="NFP weighted 4x")
axes[0].axvline(NFP, color=ALERT, linestyle="--", linewidth=1.5)
axes[0].annotate("NFP\n3 Jul", xy=(NFP, axes[0].get_ylim()[1]), xytext=(6, -28),
                 textcoords="offset points", color=ALERT, fontsize=9)
axes[0].legend(); as_percent(axes[0], decimals=1)
style_axis(axes[0], "One event date lifts the whole curve behind it", "",
           "ATM implied volatility",
           "Nothing before the event moves at all. Everything from the event onward rises, because variance is cumulative.")

axes[1].plot(merged["date"], merged["change"] * 100, color=QUATERNARY, linewidth=1.6)
axes[1].axhline(0, color=MUTED, linewidth=0.8)
axes[1].axvline(NFP, color=ALERT, linestyle="--", linewidth=1.5)
style_axis(axes[1], "", "Expiry date", "Change (vol points)",
           "The jump lands on the event date and then decays - the same extra variance spread over ever more calendar time.")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

expiries BEFORE the event that moved: 0 of 21
expiries FROM the event onward that ROSE: 99 of 99


**Result:** nothing before the event moves. The event date and **every** date after it rises. Chapter 11 calls this out as *"a real feature observed when building ATM curves"*, and the mechanism is just cumulation — the extra variance is in the bucket from that date onwards, so every later expiry inherits it.

The **decay** in the lower panel is worth noticing too. The same fixed lump of extra variance gets divided by more and more calendar time, so its effect on the quoted volatility shrinks with maturity. That is why events dominate short-dated pricing and barely register at one year.

### Experiment 6 — The strip a trader actually reads

The ATM curve hides events — the lump gets smeared across everything after it. **Daily forward variance** does not.

**Predict:** convert daily variance into a forward overnight volatility. What does the event look like there compared with the curve above?

In [12]:
fig, ax = plt.subplots(figsize=(12, 4.6))
window = event_frame[event_frame["date"] <= date(2014, 8, 1)]

colours = [ALERT if d == NFP else (MUTED if w in ("Sat", "Sun") else PRIMARY)
           for d, w in zip(window["date"], window["weekday"])]
ax.bar(window["date"], window["forward_overnight_vol"], color=colours, width=0.8)
as_percent(ax, decimals=0)
style_axis(ax, "Implied forward overnight volatility, day by day", "Date",
           "Forward overnight ATM volatility",
           "Grey bars are weekends (zero variance). The red bar is NFP. This is the view that makes an event obvious - it is invisible in the curve level.")
plt.show()

weekdays = event_frame[~event_frame["weekday"].isin(["Sat", "Sun"])]
print(f"normal weekday forward overnight vol: "
      f"{weekdays[weekdays['date'] != NFP]['forward_overnight_vol'].iloc[0]:.2%}")
print(f"NFP forward overnight vol:            "
      f"{event_frame.set_index('date').loc[NFP, 'forward_overnight_vol']:.2%}")
print(f"ratio: {np.sqrt(4.0):.2f}x  = sqrt(the weight), since vol goes as sqrt(variance)")

normal weekday forward overnight vol: 10.00%
NFP forward overnight vol:            20.00%
ratio: 2.00x  = sqrt(the weight), since vol goes as sqrt(variance)


**Result:** the event stands out immediately, and the ratio is **√4 = 2×**, not 4× — because the weight scales *variance* and volatility is its square root.

This is what Chapter 11 means by traders using forward overnight volatilities *"to determine whether the ATM curve is overpriced or underpriced over events."* You cannot see an event in the curve level; you can see nothing else in the forward strip.

And note the weekends sit at exactly zero. In a real desk model they would be small but positive — Chapter 11 says so — and seeing them pinned at zero is a reminder of which simplification you are looking at.

### Experiment 7 — Can weights create an arbitrage?

**Predict:** weights control how variance is distributed. Can any set of non-negative weights produce negative forward variance?

In [13]:
import itertools
rng = np.random.default_rng(0)

print("Random non-negative weight sets, checking for negative forward variance:\n")
for trial in range(5):
    weights = {d: float(rng.uniform(0, 5)) for d in range(7)}
    c = WeightedATMCurve(HORIZON, 0.10, weekday_weights=weights)
    bad = c.negative_forward_variance_dates(200)
    print(f"  trial {trial}: weights {[round(weights[d],2) for d in range(7)]} "
          f"-> {len(bad)} bad dates")

extreme = WeightedATMCurve(HORIZON, 0.10,
                           weekday_weights={d: 0.0 for d in range(7)} | {2: 50.0})
print(f"\n  extreme (only Wednesdays, weighted 50): "
      f"{len(extreme.negative_forward_variance_dates(200))} bad dates")

print("\nNone. Non-negative weights can only ADD variance, never remove it -")
print("so total variance is non-decreasing by construction and the arbitrage")
print("cannot arise. It is the INTERPOLATION that can break it (Experiment 1),")
print("not the weighting.")

Random non-negative weight sets, checking for negative forward variance:

  trial 0: weights [3.18, 1.35, 0.2, 0.08, 4.07, 4.56, 3.03] -> 0 bad dates
  trial 1: weights [3.65, 2.72, 4.68, 4.08, 0.01, 4.29, 0.17] -> 0 bad dates


  trial 2: weights [3.65, 0.88, 4.32, 2.71, 1.5, 2.11, 0.14] -> 0 bad dates


  trial 3: weights [0.62, 3.35, 3.24, 3.08, 1.92, 4.99, 4.9] -> 0 bad dates
  trial 4: weights [3.43, 3.25, 3.44, 1.94, 0.68, 3.61, 2.63] -> 0 bad dates

  extreme (only Wednesdays, weighted 50): 0 bad dates

None. Non-negative weights can only ADD variance, never remove it -
so total variance is non-decreasing by construction and the arbitrage
cannot arise. It is the INTERPOLATION that can break it (Experiment 1),
not the weighting.


**Result:** no set of non-negative weights can produce the arbitrage. It follows directly from the construction — cumulative sums of non-negative numbers only ever increase.

That is worth knowing precisely because it tells you **where the danger actually is**. Chapter 11 notes desks build curves "in such a way that nonnegative forward variance is guaranteed", and the day-weight layer gives that for free. The risk lives entirely in the core curve and how it interpolates — which is Task A, not Task C.

Our surface *checks* rather than guarantees, because it composes a weighted layer on top of an interpolated core. That's in `notes/deviations.md`.

## Common misconceptions

**"Calendar time and economic time are the same thing with different names."**
They are the two different denominators that make the saw-tooth exist. Variance accumulates in **economic** time; volatility is quoted against **calendar** time. Use one for both and the effect disappears — you have assumed the market trades on Sunday.

**"You can interpolate volatility directly."**
You can, and Experiment 1 shows it generating an arbitrage from valid inputs. Curves get built in variance for a reason.

**"Negative forward variance is a numerical glitch to clamp."**
It is a **calendar arbitrage**. It says a later date is less uncertain than an earlier one. Clamping it hides a broken curve rather than fixing one.

**"A Friday overnight is comparable to a Tuesday overnight."**
It is not. Chapter 11: on a Friday the "overnight" spans three days, so `T = 3/365`, and the quoted volatility must be multiplied by **√3** to compare. The book notes short-dated options sometimes get too cheap on Fridays because some banks oversell to reduce weekend theta — a live trading opportunity arising from exactly this confusion.

**"Weighting an event only affects that date."**
It affects that date and **every date after it**, because variance is cumulative. Experiment 5 measures it.

**"An event weight of 4 means the volatility is 4× higher."**
It means the *variance* is 4× higher, so the forward overnight volatility is **2×** higher. Weights scale variance; volatility is its square root.

**"The ATM curve tells you where events are."**
It smears them. The **forward overnight volatility strip** is where an event is obvious. Experiment 6 shows both views of the same curve.

## Check yourself

1. Overnight volatility is 12%. Weekdays are equally volatile and the weekend is dead. What is the 1-week volatility?
2. A 1-week option is at 12%. It is known the 8th day will be completely static. What is the 8-day volatility, and why is it a *lower bound*?
3. Your linear-in-volatility curve reports negative forward variance between 18 months and 2 years. What can you actually do about it, commercially?
4. You raise an event weight from 1 to 9. By what factor does that day's forward overnight volatility rise?
5. Why does a Monday expiry price below the Friday before it, when it is *further away* and therefore contains more uncertainty?

In [14]:
#@title Answers — run this cell to reveal
from IPython.display import Markdown
Markdown(r'''
**1.** `σ_1wk = σ_O/N × √(5/7) = 12% × 0.845 = **10.1%**`. Five live days of variance quoted against seven calendar days. The one-week is *lower* than the overnight despite covering more time.

**2.** Variance is unchanged (the 8th day adds none) but calendar time rises from 7/365 to 8/365, so `σ = 12% × √(7/8) = **11.25%**`. It is a **lower bound** because the 8th day cannot subtract variance — any real activity on it only pushes the number up. Chapter 11 makes this point explicitly: given a 7-day at 12%, the 8-day must be at least 11.25%.

**3.** Sell the 18-month option and buy the 2-year. You are paying less for *more* total variance — the market is telling you the further date is less uncertain than the nearer one, which cannot be true. In practice bid-offer will eat most or all of it, and the real answer is that your **curve is broken**: rebuild it in variance terms.

**4.** **3×.** The weight scales variance; volatility is its square root, and `√9 = 3`. The most common error here is answering 9.

**5.** Because "further away" is measured in **calendar** time and uncertainty accumulates in **economic** time. Between Friday and Monday, calendar time advances three days while economic time advances by one (Saturday and Sunday contribute nothing). Volatility is `√(variance / calendar time)` — the numerator barely moved and the denominator jumped, so the quote falls.
''')


**1.** `σ_1wk = σ_O/N × √(5/7) = 12% × 0.845 = **10.1%**`. Five live days of variance quoted against seven calendar days. The one-week is *lower* than the overnight despite covering more time.

**2.** Variance is unchanged (the 8th day adds none) but calendar time rises from 7/365 to 8/365, so `σ = 12% × √(7/8) = **11.25%**`. It is a **lower bound** because the 8th day cannot subtract variance — any real activity on it only pushes the number up. Chapter 11 makes this point explicitly: given a 7-day at 12%, the 8-day must be at least 11.25%.

**3.** Sell the 18-month option and buy the 2-year. You are paying less for *more* total variance — the market is telling you the further date is less uncertain than the nearer one, which cannot be true. In practice bid-offer will eat most or all of it, and the real answer is that your **curve is broken**: rebuild it in variance terms.

**4.** **3×.** The weight scales variance; volatility is its square root, and `√9 = 3`. The most common error here is answering 9.

**5.** Because "further away" is measured in **calendar** time and uncertainty accumulates in **economic** time. Between Friday and Monday, calendar time advances three days while economic time advances by one (Saturday and Sunday contribute nothing). Volatility is `√(variance / calendar time)` — the numerator barely moved and the denominator jumped, so the quote falls.


## Where next

**Notebook 11 — Practical F** builds the strike dimension: the Malz smile, strike-from-delta, and then the assembled surface that joins this ATM curve to a smile at every tenor.

Before moving on, try this: set the weekend weight to 0.15 instead of 0.0 — Chapter 11's "small but non-zero" — and see how much of the saw-tooth survives. That single parameter is one a real desk argues about.